# ETL Process - Weather Data (Open Meteo API)
**Bootcamp Data Analyst | Day 25 - ETL (Extract Transform Load)**


In [ ]:
!pip install requests
!pip install pandas
!pip install sqlalchemy
!pip install psycopg2-binary
!pip install python-dotenv

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
import requests
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
USER     = os.environ["DB_USER"]
PASSWORD = os.environ["DB_PASSWORD"]
HOST     = os.environ["DB_HOST"]
PORT     = os.environ["DB_PORT"]
DATABASE = os.environ["DB_NAME"]

In [40]:
def query_data(query):
    db_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
    try:
        engine = create_engine(db_url)  # Connect to database
        df = pd.read_sql(query, con=engine)  # Read data from query result
        print("Connection Successfully")
    except Exception as e:
        print(f"Error to connect: {e}")
    finally:
        # Close Connection from engine
        if "engine" in locals():
            engine.dispose()
    return df

In [41]:
def load_data(df, table_name, schema, if_exists='replace'):
    db_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
    try:
        engine = create_engine(db_url)
        with engine.connect() as conn:
            conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
            conn.commit()
        df.to_sql(table_name, engine, schema=schema, if_exists=if_exists, index=False)
        print(f"Load ke {schema}.{table_name} berhasil! ({len(df)} baris)")
    except Exception as e:
        print(f"Error load data: {e}")
    finally:
        if "engine" in locals():
            engine.dispose()

In [42]:
def extract_api(url):
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    print('Data berhasil di extract')
    return data

In [43]:
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-6.2088"
    "&longitude=106.8456"
    "&hourly=temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,weather_code"
    "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunrise,sunset"
    "&timezone=Asia%2FJakarta"
    "&forecast_days=7"
)

data = extract_api(url)

Data berhasil di extract


In [44]:
# Hourly
df = pd.DataFrame(data['hourly'])
df['latitude']  = data['latitude']
df['longitude'] = data['longitude']
df['city']      = 'Jakarta'

print(f'Jumlah baris hourly: {len(df)}')
df.head()

Jumlah baris hourly: 168


,time,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,weather_code,latitude,longitude,city
0,2026-07-07T00:00,26.6,79,18,0.0,4.5,0,-6.221441,106.856186,Jakarta
1,2026-07-07T01:00,26.9,79,4,0.0,1.5,0,-6.221441,106.856186,Jakarta
2,2026-07-07T02:00,26.7,82,0,0.0,2.7,0,-6.221441,106.856186,Jakarta
3,2026-07-07T03:00,26.4,84,0,0.0,3.1,0,-6.221441,106.856186,Jakarta
4,2026-07-07T04:00,26.1,87,0,0.0,3.6,0,-6.221441,106.856186,Jakarta


In [45]:
# Daily
df_daily = pd.DataFrame(data['daily'])
df_daily['city'] = 'Jakarta'

print(f'Jumlah baris daily: {len(df_daily)}')
df_daily.head()

Jumlah baris daily: 7


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunrise,sunset,city
0,2026-07-07,33.4,25.6,1.4,9.9,2026-07-07T06:04,2026-07-07T17:50,Jakarta
1,2026-07-08,33.8,25.7,0.0,13.8,2026-07-08T06:04,2026-07-08T17:50,Jakarta
2,2026-07-09,34.9,25.1,0.0,14.0,2026-07-09T06:04,2026-07-09T17:51,Jakarta
3,2026-07-10,34.4,25.2,0.5,14.3,2026-07-10T06:04,2026-07-10T17:51,Jakarta
4,2026-07-11,31.9,24.8,1.4,13.3,2026-07-11T06:04,2026-07-11T17:51,Jakarta


In [46]:
print('=== INFO DATAFRAME ===')
df.info()
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

=== INFO DATAFRAME ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   time                       168 non-null    object 
 1   temperature_2m             168 non-null    float64
 2   relative_humidity_2m       168 non-null    int64  
 3   precipitation_probability  168 non-null    int64  
 4   precipitation              168 non-null    float64
 5   wind_speed_10m             168 non-null    float64
 6   weather_code               168 non-null    int64  
 7   latitude                   168 non-null    float64
 8   longitude                  168 non-null    float64
 9   city                       168 non-null    object 
dtypes: float64(5), int64(3), object(2)
memory usage: 13.3+ KB

=== MISSING VALUES ===
time                         0
temperature_2m               0
relative_humidity_2m         0
precipitation_probab

In [47]:
WEATHER_CODE_MAP = {
    0:  'Clear sky',
    1:  'Mainly clear',
    2:  'Partly cloudy',
    3:  'Overcast',
    45: 'Fog',
    48: 'Rime fog',
    51: 'Light drizzle',
    53: 'Moderate drizzle',
    55: 'Dense drizzle',
    61: 'Slight rain',
    63: 'Moderate rain',
    65: 'Heavy rain',
    80: 'Slight showers',
    81: 'Moderate showers',
    82: 'Violent showers',
    95: 'Thunderstorm'
}

df_clean = df.copy()

# 1. Standarisasi tipe data
df_clean['time'] = pd.to_datetime(df_clean['time'])

# 2. Derivasi kolom baru
df_clean['date']       = df_clean['time'].dt.date
df_clean['hour']       = df_clean['time'].dt.hour
df_clean['is_daytime'] = df_clean['hour'].between(6, 18).astype(int)

# 3. Mapping weather_code ke deskripsi
df_clean['weather_description'] = df_clean['weather_code'].map(WEATHER_CODE_MAP).fillna('Unknown')

# 4. Kategorisasi suhu
df_clean['temp_category'] = pd.cut(
    df_clean['temperature_2m'],
    bins=[-999, 20, 28, 35, 999],
    labels=['Dingin', 'Nyaman', 'Hangat', 'Panas']
)

# 5. Handle missing values
df_clean['precipitation']             = df_clean['precipitation'].fillna(0)
df_clean['precipitation_probability'] = df_clean['precipitation_probability'].fillna(0)
df_clean['wind_speed_10m']            = df_clean['wind_speed_10m'].fillna(df_clean['wind_speed_10m'].median())

# 6. Hapus duplikat
df_clean = df_clean.drop_duplicates(subset=['time', 'city'])

# 7. Metadata
df_clean['etl_loaded_at'] = pd.Timestamp.now()

print('Transform hourly selesai!')
df_clean.head()

Transform hourly selesai!


,time,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,weather_code,latitude,longitude,city,date,hour,is_daytime,weather_description,temp_category,etl_loaded_at
0,2026-07-07 00:00:00,26.6,79,18,0.0,4.5,0,-6.221441,106.856186,Jakarta,2026-07-07,0,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
1,2026-07-07 01:00:00,26.9,79,4,0.0,1.5,0,-6.221441,106.856186,Jakarta,2026-07-07,1,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
2,2026-07-07 02:00:00,26.7,82,0,0.0,2.7,0,-6.221441,106.856186,Jakarta,2026-07-07,2,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
3,2026-07-07 03:00:00,26.4,84,0,0.0,3.1,0,-6.221441,106.856186,Jakarta,2026-07-07,3,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
4,2026-07-07 04:00:00,26.1,87,0,0.0,3.6,0,-6.221441,106.856186,Jakarta,2026-07-07,4,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539


In [48]:
df_daily_clean = df_daily.copy()

df_daily_clean['time']    = pd.to_datetime(df_daily_clean['time'])
df_daily_clean['sunrise'] = pd.to_datetime(df_daily_clean['sunrise'])
df_daily_clean['sunset']  = pd.to_datetime(df_daily_clean['sunset'])

df_daily_clean['daylight_hours'] = (
    (df_daily_clean['sunset'] - df_daily_clean['sunrise'])
    .dt.total_seconds() / 3600
).round(2)

df_daily_clean['temp_range'] = (
    df_daily_clean['temperature_2m_max'] - df_daily_clean['temperature_2m_min']
).round(2)

df_daily_clean['rain_category'] = pd.cut(
    df_daily_clean['precipitation_sum'],
    bins=[-1, 0, 5, 20, 999],
    labels=['Tidak Hujan', 'Hujan Ringan', 'Hujan Sedang', 'Hujan Lebat']
)

df_daily_clean['precipitation_sum'] = df_daily_clean['precipitation_sum'].fillna(0)
df_daily_clean['etl_loaded_at']     = pd.Timestamp.now()

print('Transform daily selesai!')
df_daily_clean.head()

Transform daily selesai!


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunrise,sunset,city,daylight_hours,temp_range,rain_category,etl_loaded_at
0,2026-07-07,33.4,25.6,1.4,9.9,2026-07-07 06:04:00,2026-07-07 17:50:00,Jakarta,11.77,7.8,Hujan Ringan,2026-07-07 10:10:18.777656
1,2026-07-08,33.8,25.7,0.0,13.8,2026-07-08 06:04:00,2026-07-08 17:50:00,Jakarta,11.77,8.1,Tidak Hujan,2026-07-07 10:10:18.777656
2,2026-07-09,34.9,25.1,0.0,14.0,2026-07-09 06:04:00,2026-07-09 17:51:00,Jakarta,11.78,9.8,Tidak Hujan,2026-07-07 10:10:18.777656
3,2026-07-10,34.4,25.2,0.5,14.3,2026-07-10 06:04:00,2026-07-10 17:51:00,Jakarta,11.78,9.2,Hujan Ringan,2026-07-07 10:10:18.777656
4,2026-07-11,31.9,24.8,1.4,13.3,2026-07-11 06:04:00,2026-07-11 17:51:00,Jakarta,11.78,7.1,Hujan Ringan,2026-07-07 10:10:18.777656


In [49]:
print('=== Missing values setelah transform ===')
print(df_clean.isnull().sum())

=== Missing values setelah transform ===
time                         0
temperature_2m               0
relative_humidity_2m         0
precipitation_probability    0
precipitation                0
wind_speed_10m               0
weather_code                 0
latitude                     0
longitude                    0
city                         0
date                         0
hour                         0
is_daytime                   0
weather_description          0
temp_category                0
etl_loaded_at                0
dtype: int64


In [50]:
print('=== INITIAL LOAD ===')

# Staging — data mentah
load_data(df,       'stg_weather_hourly', schema='staging', if_exists='replace')
load_data(df_daily, 'stg_weather_daily',  schema='staging', if_exists='replace')

# Warehouse — data bersih
load_data(df_clean,       'fact_weather_hourly', schema='warehouse', if_exists='replace')
load_data(df_daily_clean, 'fact_weather_daily',  schema='warehouse', if_exists='replace')

print('\nInitial Load selesai!')

=== INITIAL LOAD ===
Load ke staging.stg_weather_hourly berhasil! (168 baris)
Load ke staging.stg_weather_daily berhasil! (7 baris)
Load ke warehouse.fact_weather_hourly berhasil! (168 baris)
Load ke warehouse.fact_weather_daily berhasil! (7 baris)

Initial Load selesai!


In [51]:
print('=== DELTA LOAD ===')

query = """
    select max(time) as last_time
    from warehouse.fact_weather_hourly
"""
df_last   = query_data(query)
last_time = df_last.iloc[0, 0]
print(f'Data terakhir di warehouse: {last_time}')

df_clean['time'] = pd.to_datetime(df_clean['time'])
df_delta         = df_clean[df_clean['time'] > pd.Timestamp(last_time)]

if len(df_delta) == 0:
    print('Tidak ada data baru. Delta Load selesai.')
else:
    load_data(df_delta, 'fact_weather_hourly', schema='warehouse', if_exists='append')
    print(f'Delta Load selesai! {len(df_delta)} baris baru ditambahkan.')

=== DELTA LOAD ===
Connection Successfully
Data terakhir di warehouse: 2026-07-13 23:00:00
Tidak ada data baru. Delta Load selesai.


In [52]:
query = """
    select *
    from warehouse.fact_weather_hourly
    limit 10
"""
df_result = query_data(query)
df_result.head()

Connection Successfully


,time,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,weather_code,latitude,longitude,city,date,hour,is_daytime,weather_description,temp_category,etl_loaded_at
0,2026-07-07 00:00:00,26.6,79,18,0.0,4.5,0,-6.221441,106.856186,Jakarta,2026-07-07,0,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
1,2026-07-07 01:00:00,26.9,79,4,0.0,1.5,0,-6.221441,106.856186,Jakarta,2026-07-07,1,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
2,2026-07-07 02:00:00,26.7,82,0,0.0,2.7,0,-6.221441,106.856186,Jakarta,2026-07-07,2,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
3,2026-07-07 03:00:00,26.4,84,0,0.0,3.1,0,-6.221441,106.856186,Jakarta,2026-07-07,3,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
4,2026-07-07 04:00:00,26.1,87,0,0.0,3.6,0,-6.221441,106.856186,Jakarta,2026-07-07,4,0,Clear sky,Nyaman,2026-07-07 10:10:18.725539
